# 🎙️ AI Voice Clone Detector — End-to-End Google Colab Pipeline

This notebook provides a complete one-click pipeline to:
1. Mount Google Drive and check GPU acceleration
2. Install dependencies & clone/setup the repository
3. Build official manifests (ASVspoof 2019 LA / In-the-Wild / Custom folder)
4. Precompute & cache MFCC / Log-Mel audio features
5. Train the `VoiceCloneCNN` model with class-weighted loss, LR scheduler, and early stopping
6. Evaluate model performance (EER, AUC, Per-Attack analysis)
7. Generate publication-quality figures (ROC, DET, Confusion Matrix, Score Histograms)
8. Launch the interactive Gradio Web Dashboard with a public shareable URL

## 1. Hardware Check & Google Drive Mount

In [ ]:
# Verify GPU availability
!nvidia-smi

# Mount Google Drive to persist checkpoints and access dataset
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

## 2. Environment Setup & Dependency Installation

In [ ]:
import os

# Define workspace path
REPO_DIR = "/content/voice-clone-detector"
if not os.path.exists(REPO_DIR):
    # If cloning from Git or copying from Drive:
    # !git clone https://github.com/Abinav-N/voice-clone-detector.git /content/voice-clone-detector
    os.makedirs(REPO_DIR, exist_ok=True)

%cd {REPO_DIR}

# Install project requirements
!pip install -q torch torchaudio librosa soundfile scikit-learn scipy tqdm gradio matplotlib pytest
print("✓ All dependencies installed successfully!")

## 3. Run Test Suite (Sanity Check)

In [ ]:
# Verify all feature extraction, metrics, and models pass unit tests
!pytest tests/ -v

## 4. Build Manifests

Point to your dataset path in Google Drive (e.g., `/content/drive/MyDrive/asvspoof2019` or `/content/drive/MyDrive/for-original`).

In [ ]:
# Example 1: Building manifests from folder with real/ and fake/ subdirectories
# !python src/build_manifest.py --dataset folder --root /content/drive/MyDrive/for-original --out manifests/

# Example 2: Official ASVspoof 2019 LA benchmark
# !python src/build_manifest.py --dataset asvspoof --root /content/drive/MyDrive/ASVspoof2019 --out manifests/

# Example 3: In-the-Wild dataset
# !python src/build_manifest.py --dataset itw --root /content/drive/MyDrive/in_the_wild --out manifests/

!ls -lh manifests/

## 5. Train VoiceCloneCNN Model

Features:
- `--feature mfcc` or `--feature logmel`
- `--weighted-loss` to combat class imbalance
- `--patience 5` for early stopping when validation EER plateaus
- `--cache` to precompute & store `.npy` features for 10x faster training epochs

In [ ]:
!python src/train.py \
    --train manifests/train.csv \
    --val manifests/val.csv \
    --epochs 25 \
    --batch-size 32 \
    --lr 0.001 \
    --feature mfcc \
    --weighted-loss \
    --patience 5 \
    --cache \
    --cache-dir /content/cache_features \
    --out checkpoints/

## 6. Comprehensive Evaluation & Per-Attack Analysis

In [ ]:
!python src/evaluate.py \
    --checkpoint checkpoints/best.pt \
    --manifest manifests/eval.csv \
    --name asvspoof_eval \
    --results-dir results/

## 7. Generate Evaluation Figures & Curves

In [ ]:
# Generates: ROC curve, DET curve, Confusion Matrix, Score Histogram, Training Curves
!python src/plots.py \
    --scores results/asvspoof_eval.csv \
    --history checkpoints/history.csv \
    --out-dir results/figures

# Display generated figures
from IPython.display import Image, display
import glob
for img in sorted(glob.glob("results/figures/*.png")):
    print(f"\nFigure: {img}")
    display(Image(filename=img, width=600))

## 8. Launch Interactive Gradio Dashboard (Public Share Link)

Includes:
- 🔍 **Classify Single Clip** (waveform, spectrogram, and Grad-CAM explainability)
- 📦 **Batch Classify** with dynamic threshold slider and CSV export
- 📊 **Results & Performance Dashboard**

In [ ]:
# Pass --share to obtain a public gradio.live URL
!python app.py --share --checkpoint checkpoints/best.pt